In [ ]:
# I wrote the explanation about changed parts
# in this source code with comments.

In [ ]:
# From the beginning here to visualizing the original data scatter plot,
# it is the same as the code of HW2.
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
plt.style.use("ggplot")
%matplotlib inline

In [ ]:
from pylab import rcParams
rcParams['figure.figsize'] = 12, 8

In [ ]:
import pandas as pd
import io
from google.colab import files

uploaded = files.upload()
data=pd.read_csv("data.csv",index_col="t")
data.head()

In [ ]:
feature = data[["x1", "x2"]].values
class_label = data.index.values

In [ ]:
class_c0 = (class_label == 1).reshape(100, 1)
class_c1 = (class_label == 0).reshape(100, 1)

ax = sns.scatterplot(x = feature[class_c0[:, 0], 0],
                     y = feature[class_c0[:, 0], 1],
                     marker = "^",
                     color = "green",
                     s = 60)
sns.scatterplot(x = feature[class_c1[:, 0], 0],
                y = feature[class_c1[:, 0], 1],
                marker = "X",
                color = "red",
                s = 60)

ax.set(xlabel="x1", ylabel="x2")
ax.legend(["C0", "C1"])
plt.show();

In [ ]:
# I changed the name of the pre-defined 'logistic_function'
#  into 'sigmoid' which is necessary to use in the output layer.
def sigmoid(x):
    return 1/ (1 + np.exp(-x))

# I defined 'relu' function for hidden layer.
def relu(x):
    return np.maximum(0, x)

# I added the function to compute derivative of relu function.
def relu_derivative(x):
    return np.where(x > 0, 1, 0)

In [ ]:
# Unlike HW2, I separated bias terms (b1 and b2) from weights (theta1 and theta2).
# So, the shape of x is (m, 2) not (m, 3).

# The shape of variables
# x :       (m, 2)
# h :       (m, 3)
# y_pred :  m
# y :       m
#------------------
# theta1 :  (2, 3)
# b1 :      3
# theta2 :  (3, 1)
# b2 :      1
def compute_cost(params, x, y, epsilon=1e-8):
    # The input 'params' and the output 'gradients' are list-typed variables.
    # params contains 4 parameters: theta1, b1, theta2, b2.
    # gradients contains 4 gradients of loss with respect to each parameter.
    # epsilon is used in computing error.
    theta1, b1, theta2, b2 = params

    # m is the number of input samples
    m = len(y)

    # h is hidden layer
    # z1 and z2 are the values before inserted in activation functions.
    z1 = np.dot(x, theta1) + b1
    h = relu(z1)

    z2 = np.dot(h, theta2) + b2
    y_pred = sigmoid(z2)

    # The sufficiently small number epsilon is used to prevent the inner part of log of error from being 0.
    error = (y * np.log(y_pred + epsilon)) + ((1 - y) * np.log(1 - y_pred + epsilon))
    cost = -1 / m * sum(error)

    # To compute gradient, I implemented backpropagation step by step using Chain rule.
    # The variable d_z2 means the partial derivative d_J/d_z2.
    # d_theta2 = d_J/d_theta2, d_b2 = d_J/d_b2, ...
    d_z2 = 1 / m * (y_pred - y)                 # shape: (m, 1)
    d_theta2 = np.dot(h.T, d_z2)                # shape: (3, 1)
    d_b2 = np.sum(d_z2, axis=0, keepdims=True)  # shape: (1, 1)
    d_h = np.dot(d_z2, theta2.T)                # shape: (m, 3)
    d_z1 = d_h * relu_derivative(h)             # shape: (m, 3)
    d_theta1 = np.dot(x.T, d_z1)                # shape: (2, 3)
    d_b1 = np.sum(d_z1, axis=0, keepdims=True)  # shape: (1, 3)

    gradients = [d_theta1, d_b1, d_theta2, d_b2]
    return cost[0] , gradients

In [ ]:
# I choose to use He normal initializtion
# which is suitable for relu activation function.

# I fixed random seed to generate the same and optimal result for every run.
# This ensures the same weights initialization for every run.
np.random.seed(50)

# He Normal Initialization
def he_init(n_in, n_out):
    return np.random.randn(n_in, n_out) * np.sqrt(2 / n_in)

In [ ]:
mean_scores = np.mean(feature, axis=0)
std_scores = np.std(feature, axis=0)
scores = (feature - mean_scores) / std_scores #standardization

rows = feature.shape[0]
cols = feature.shape[1]

# Unlike HW2, I didn't include intercept column in X
# because I separated bias terms (b1 and b2) from weights (theta1 and theta2).
X = scores
y = class_label.reshape(rows, 1)

# These three variables indicate the dimension of each layer
input_dim = X.shape[1]  # input_dim = 2
hidden_dim = 3
output_dim = 1

# Initialize parameters
# Create parameters theta1 and theta2 using He initialization
# Create biases b1 and b2, and initialize them with just 0s.
theta1 = he_init(input_dim, hidden_dim)
b1 = np.zeros((1, hidden_dim))
theta2 = he_init(hidden_dim, output_dim)
b2 = np.zeros((1, output_dim))
params_init = [theta1, b1, theta2, b2]

cost, gradients = compute_cost(params_init, X, y)

print("Cost at initialization", cost)
print("Gradient at initialization:", gradients)

In [ ]:
# I defined the function 'adam' to update parameters.
def adam(params, gradients, m, v, alpha, beta1=0.9, beta2=0.999, epsilon=1e-8):
    # For each parameter (theta1, b1, theta2, b2), compute m, v, m_hat, and v_hat.
    # Then, update the parameters to the extent of alpha.
    for i in range(len(params)):
        m[i] = beta1 * m[i] + (1 - beta1) * gradients[i]
        v[i] = beta2 * v[i] + (1 - beta2) * np.square(gradients[i])
        m_hat = m[i] / (1 - beta1)
        v_hat = v[i] / (1 - beta2)
        params[i] -= alpha * m_hat / (np.sqrt(v_hat) + epsilon)

    return params, m, v

In [ ]:
def gradient_descent(x, y, params, alpha, iterations):
    costs = []

    # Initialize m0 and v0 to all zeros matrices
    m = [np.zeros_like(p) for p in params]
    v = [np.zeros_like(p) for p in params]

    # For every iteration, I updated parameters using the function 'adam'
    for i in range(iterations):
        cost, gradients = compute_cost(params, x, y)
        params, m, v = adam(params, gradients, m, v, alpha)
        costs.append(cost)
    return params, costs

In [ ]:
params, costs = gradient_descent(X, y, params_init, 1, 200)

In [ ]:
print("Theta after running gradient descent:", params)
print("Resulting cost:", costs[-1])

In [ ]:
plt.plot(costs)
plt.xlabel("Iterations")
plt.ylabel("$J(\Theta)$")
plt.title("Values of Cost Function over iterations of Gradient Descent");

In [ ]:
# First, draw the scatter plots for points of class 0 and 1.
sns.scatterplot(x = X[class_c0[:, 0], 0],
                y = X[class_c0[:, 0], 1],
                marker = "^",
                color = "green",
                s = 60)
ax = sns.scatterplot(x = X[class_c1[:, 0], 0],
                    y = X[class_c1[:, 0], 1],
                    marker = "X",
                    color = "red",
                    s = 60)

ax.legend(["C0", "C1"])
ax.set(xlabel="x1", ylabel="x2")

# Prepare for drawing decision boundary.
# X_grid contains almost every points (with resolution 0.02) of the coordinate.
resolution = 0.02
xs = np.arange(np.min(X[:, 0])-0.2, np.max(X[:, 0])+0.2, resolution)
ys = np.arange(np.min(X[:, 1])-0.2, np.max(X[:, 1])+0.2, resolution)
xx, yy = np.meshgrid(xs, ys)

X_grid = np.column_stack((xx.ravel(), yy.ravel()))

# The variable 'params' contains the best parameters after training the model for 200 iterations
theta1, b1, theta2, b2 = params

Z1 = np.dot(X_grid, theta1) + b1
Z2 = np.dot(relu(Z1), theta2) + b2
Z = sigmoid(Z2)
Z = Z.reshape(xx.shape)

# Draw a curve where Z = 0.5.
ax.contour(xx, yy, Z, [0.5], colors='blue');

plt.show()

In [ ]:
def predict(params, x):
    # Using parameters after training the model for 200 iterations,
    # compute z2 and return whether z2 is bigger than 0.
    # If z2 > 0, the probability for the sample x of class 1 is larger than 0.5.
    # Then, the model predicts x as of class 1.
    theta1, b1, theta2, b2 = params

    z1 = np.dot(x, theta1) + b1
    z2 = np.dot(relu(z1), theta2) + b2
    return z2 > 0


In [ ]:
p = predict(params, X)
print("Training Accuracy:", sum(p==y)[0],"%")

In [ ]:
test = np.array([50,79])
test = (test - mean_scores)/std_scores

theta1, b1, theta2, b2 = params
Z1 = np.dot(test, theta1) + b1
Z2 = np.dot(relu(Z1), theta2) + b2

probability = sigmoid(Z2)
print("A test sample who x1 = 50 and x2 = 79 will be in the class of C0 in the probability of ",
      np.round(probability[0], 2))